In [1]:
# -------------------------------
# Imports
# -------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# -------------------------------
# Dataset class with augmentation
# -------------------------------
class PriceDataset(Dataset):
    def __init__(self, X_img, X_txt, y=None, augment=False):
        self.X_img = torch.FloatTensor(X_img)
        self.X_txt = torch.FloatTensor(X_txt)
        self.y = torch.FloatTensor(y) if y is not None else None
        self.augment = augment

    def __len__(self):
        return len(self.X_img)

    def __getitem__(self, idx):
        img, txt = self.X_img[idx], self.X_txt[idx]

        # Data augmentation for training
        if self.augment and self.y is not None:
            # Add small Gaussian noise
            if np.random.rand() > 0.5:
                img = img + torch.randn_like(img) * 0.01
                txt = txt + torch.randn_like(txt) * 0.01

        if self.y is not None:
            return img, txt, self.y[idx]
        return img, txt

# -------------------------------
# Multi-Head Cross-Attention with Gating
# -------------------------------
class GatedCrossAttention(nn.Module):
    def __init__(self, dim_img, dim_txt, dim_hidden, num_heads=8, dropout=0.2):
        super().__init__()
        self.num_heads = num_heads
        self.dim_hidden = dim_hidden
        self.head_dim = dim_hidden // num_heads

        self.query_img = nn.Linear(dim_img, dim_hidden)
        self.key_txt = nn.Linear(dim_txt, dim_hidden)
        self.value_txt = nn.Linear(dim_txt, dim_hidden)
        self.key_img = nn.Linear(dim_img, dim_hidden)
        self.value_img = nn.Linear(dim_img, dim_hidden)

        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        # Gating mechanism
        self.gate = nn.Sequential(
            nn.Linear(dim_img + dim_txt, dim_hidden),
            nn.Sigmoid()
        )

        self.out_proj = nn.Linear(dim_hidden * 2, dim_hidden)
        self.layer_norm = nn.LayerNorm(dim_hidden)

    def forward(self, img, txt):
        batch_size = img.size(0)

        # Cross-attention: image queries attend to text
        Q_img = self.query_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K_txt = self.key_txt(txt).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V_txt = self.value_txt(txt).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        attn_txt = torch.softmax(torch.matmul(Q_img, K_txt.transpose(-2, -1)) * self.scale, dim=-1)
        attn_txt = self.dropout(attn_txt)
        cross_txt = torch.matmul(attn_txt, V_txt).transpose(1, 2).contiguous().view(batch_size, self.dim_hidden)

        # Self-attention on image
        K_img = self.key_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V_img = self.value_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        attn_img = torch.softmax(torch.matmul(Q_img, K_img.transpose(-2, -1)) * self.scale, dim=-1)
        attn_img = self.dropout(attn_img)
        self_img = torch.matmul(attn_img, V_img).transpose(1, 2).contiguous().view(batch_size, self.dim_hidden)

        # Gated fusion
        gate_input = torch.cat([img, txt], dim=-1)
        gate_weights = self.gate(gate_input)

        # Combine both attention outputs
        combined = torch.cat([cross_txt, self_img], dim=-1)
        fused = self.out_proj(combined)
        fused = gate_weights * fused

        return self.layer_norm(fused)

# -------------------------------
# Advanced Residual Block with SE-Net
# -------------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        # Squeeze
        w = torch.mean(x, dim=0, keepdim=True)
        # Excitation
        w = torch.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout)
        )
        self.se = SEBlock(dim)
        self.layer_norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.net(x)
        x = self.se(x)
        return self.layer_norm(x + residual)

# -------------------------------
# Advanced Model Architecture
# -------------------------------
class AdvancedPriceModel(nn.Module):
    def __init__(self, dim_img, dim_txt, hidden_dims=[1536, 768, 384], dropout=0.15, num_heads=12):
        super().__init__()

        # Input projections with layer norm
        self.img_proj = nn.Sequential(
            nn.Linear(dim_img, hidden_dims[0] // 2),
            nn.LayerNorm(hidden_dims[0] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )

        self.txt_proj = nn.Sequential(
            nn.Linear(dim_txt, hidden_dims[0] // 2),
            nn.LayerNorm(hidden_dims[0] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )

        # Cross-attention fusion
        self.cross_attn = GatedCrossAttention(
            hidden_dims[0] // 2,
            hidden_dims[0] // 2,
            hidden_dims[0],
            num_heads=num_heads,
            dropout=dropout
        )

        # Stack of residual blocks
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_dims[0], dropout) for _ in range(3)
        ])

        # Progressive dimensionality reduction with skip connections
        self.down_blocks = nn.ModuleList()
        prev_dim = hidden_dims[0]
        for h_dim in hidden_dims[1:]:
            self.down_blocks.append(nn.Sequential(
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ))
            prev_dim = h_dim

        # Multi-scale features
        self.skip_connections = nn.ModuleList([
            nn.Linear(hidden_dims[0], hidden_dims[-1])
        ])

        # Final prediction head
        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dims[-1] * 2, hidden_dims[-1]),
            nn.LayerNorm(hidden_dims[-1]),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dims[-1], hidden_dims[-1] // 2),
            nn.LayerNorm(hidden_dims[-1] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.25),
            nn.Linear(hidden_dims[-1] // 2, 1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, img, txt):
        # Project inputs
        img_feat = self.img_proj(img)
        txt_feat = self.txt_proj(txt)

        # Cross-attention fusion
        x = self.cross_attn(img_feat, txt_feat)

        # Store for skip connection
        skip = x

        # Apply residual blocks
        for block in self.residual_blocks:
            x = block(x)

        # Progressive downsampling
        for down in self.down_blocks:
            x = down(x)

        # Multi-scale fusion
        skip_transformed = self.skip_connections[0](skip)
        x = torch.cat([x, skip_transformed], dim=-1)

        # Final prediction
        out = self.prediction_head(x)
        return out.squeeze()

# -------------------------------
# Advanced loss function
# -------------------------------
class AdaptiveLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.mae = nn.L1Loss()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        mae_loss = self.mae(pred, target)
        return self.alpha * mse_loss + (1 - self.alpha) * mae_loss

# -------------------------------
# MSAPE metric
# -------------------------------
def msape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    epsilon = 1e-10
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2 + epsilon
    return 100 / len(y_true) * np.sum(np.abs(y_true - y_pred) / denominator)

# -------------------------------
# Advanced Training with SWA and Mixup
# -------------------------------
def train_model_advanced(model, train_loader, val_loader, device, epochs=250, lr_max=3e-4,
                        patience=40, warmup_epochs=15, log_interval=5, min_epochs=80):
    criterion = AdaptiveLoss(alpha=0.7)
    optimizer = optim.AdamW(model.parameters(), lr=lr_max, weight_decay=5e-5, betas=(0.9, 0.999))

    # Cosine annealing with warm restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-7
    )

    # SWA (Stochastic Weight Averaging)
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = min_epochs
    swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=1e-5)

    scaler = torch.amp.GradScaler(device=device)

    best_val_msape = float('inf')
    best_model_state = None
    wait = 0

    history = {'train_loss': [], 'val_loss': [], 'val_msape': [], 'lr': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        # Linear warmup
        if epoch < warmup_epochs:
            current_lr = lr_max * (epoch + 1) / warmup_epochs
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr

        for X_img_batch, X_txt_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            X_img_batch = X_img_batch.to(device)
            X_txt_batch = X_txt_batch.to(device)
            y_batch = y_batch.to(device)

            # Mixup augmentation (randomly blend samples)
            if np.random.rand() > 0.7 and epoch > warmup_epochs:
                lam = np.random.beta(0.2, 0.2)
                rand_index = torch.randperm(X_img_batch.size(0)).to(device)
                X_img_batch = lam * X_img_batch + (1 - lam) * X_img_batch[rand_index]
                X_txt_batch = lam * X_txt_batch + (1 - lam) * X_txt_batch[rand_index]
                y_batch = lam * y_batch + (1 - lam) * y_batch[rand_index]

            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type):
                outputs = model(X_img_batch, X_txt_batch)
                loss = criterion(outputs, y_batch)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []

        with torch.no_grad():
            for X_img_batch, X_txt_batch, y_batch in val_loader:
                X_img_batch = X_img_batch.to(device)
                X_txt_batch = X_txt_batch.to(device)
                y_batch = y_batch.to(device)

                with torch.amp.autocast(device_type=device.type):
                    outputs = model(X_img_batch, X_txt_batch)
                    loss = criterion(outputs, y_batch)

                val_loss += loss.item()
                val_preds.append(outputs.cpu().numpy())
                val_targets.append(y_batch.cpu().numpy())

        val_loss /= len(val_loader)
        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)
        val_msape_score = msape(np.expm1(val_targets), np.expm1(val_preds))

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_msape'].append(val_msape_score)
        history['lr'].append(current_lr)

        # Logging
        if (epoch + 1) % log_interval == 0 or epoch == 0:
            print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
                  f"Val MSAPE={val_msape_score:.4f}, LR={current_lr:.6f}")
        else:
            print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, LR={current_lr:.6f}")

        # Update SWA
        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        elif epoch >= warmup_epochs:
            scheduler.step()

        # Early stopping
        if val_msape_score < best_val_msape and epoch >= min_epochs:
            best_val_msape = val_msape_score
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
            print(f"  ✓ New best model! MSAPE: {best_val_msape:.4f}")
        else:
            wait += 1
            if wait >= patience and epoch >= min_epochs:
                print(f"Early stopping at epoch {epoch+1}")
                break
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f"/content/drive/MyDrive/checkpoints/model_epoch_{epoch+1}.pt")


    # Load best model or use SWA model
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    # Update batch norm statistics for SWA
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)

    print(f"\n✅ Best model MSAPE: {best_val_msape:.4f}")

    return model, swa_model, history

# -------------------------------
# Prepare data with robust scaling
# -------------------------------
X_image = np.array(list(df['image_encoding'].values), dtype=np.float32)
X_text = np.array(list(df['text_encoding'].values), dtype=np.float32)

# Use RobustScaler (more resistant to outliers)
scaler_img = RobustScaler()
scaler_txt = RobustScaler()
X_image = scaler_img.fit_transform(X_image)
X_text = scaler_txt.fit_transform(X_text)

y = df['price'].values.astype(np.float32)
y_log = np.log1p(y)

X_img_train, X_img_val, X_txt_train, X_txt_val, y_train, y_val = \
    train_test_split(X_image, X_text, y_log, test_size=0.12, random_state=42, shuffle=True)

batch_size = 128  # Smaller batch for better generalization
train_loader = DataLoader(
    PriceDataset(X_img_train, X_txt_train, y_train, augment=True),
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
val_loader = DataLoader(
    PriceDataset(X_img_val, X_txt_val, y_val),
    batch_size=batch_size * 2,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# -------------------------------
# Initialize model
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedPriceModel(
    dim_img=X_image.shape[1],
    dim_txt=X_text.shape[1],
    hidden_dims=[1536, 768, 384],
    dropout=0.15,
    num_heads=12
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# -------------------------------
# Train
# -------------------------------
model, swa_model, history = train_model_advanced(
    model, train_loader, val_loader, device,
    epochs=250,
    lr_max=3e-4,
    patience=40,
    warmup_epochs=15,
    log_interval=5,
    min_epochs=80
)

# -------------------------------
# Evaluate both models and use the best
# -------------------------------
def evaluate_model(eval_model, X_img, X_txt, y_true_log, device, batch_size=1024):
    eval_model.eval()
    predictions = []

    with torch.no_grad():
        X_img_tensor = torch.FloatTensor(X_img).to(device)
        X_txt_tensor = torch.FloatTensor(X_txt).to(device)

        for i in range(0, len(X_img_tensor), batch_size):
            batch_img = X_img_tensor[i:i+batch_size]
            batch_txt = X_txt_tensor[i:i+batch_size]
            with torch.amp.autocast(device_type=device.type):
                pred = eval_model(batch_img, batch_txt).cpu().numpy()
            predictions.append(pred)

    predictions = np.concatenate(predictions)
    y_pred = np.expm1(predictions)
    y_true = np.expm1(y_true_log)

    return msape(y_true, y_pred), y_pred

print("\n" + "="*60)
print("Evaluating standard model...")
msape_standard, pred_standard = evaluate_model(model, X_img_val, X_txt_val, y_val, device)
print(f"Standard Model MSAPE: {msape_standard:.4f}")

print("\nEvaluating SWA model...")
msape_swa, pred_swa = evaluate_model(swa_model, X_img_val, X_txt_val, y_val, device)
print(f"SWA Model MSAPE: {msape_swa:.4f}")

# Use the better model
if msape_swa < msape_standard:
    final_model = swa_model
    final_msape = msape_swa
    final_preds = pred_swa
    print("\n✅ Using SWA model (better performance)")
else:
    final_model = model
    final_msape = msape_standard
    final_preds = pred_standard
    print("\n✅ Using standard model (better performance)")

# Ensemble prediction (average both models)
y_pred_ensemble = (pred_standard + pred_swa) / 2
msape_ensemble = msape(np.expm1(y_val), y_pred_ensemble)
print(f"Ensemble MSAPE: {msape_ensemble:.4f}")

if msape_ensemble < final_msape:
    final_preds = y_pred_ensemble
    final_msape = msape_ensemble
    print("✅ Using ensemble predictions (best performance)")

print("="*60)
print(f"✅ Final MSAPE: {final_msape:.4f}")
print("="*60)

# Additional metrics
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
y_true = np.expm1(y_val)
mae = mean_absolute_error(y_true, final_preds)
rmse = np.sqrt(mean_squared_error(y_true, final_preds))
r2 = r2_score(y_true, final_preds)
mape = np.mean(np.abs((y_true - final_preds) / (y_true + 1e-10))) * 100

print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"Root Mean Squared Error: ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

# -------------------------------
# Save everything
# -------------------------------
torch.save({
    'model_state_dict': model.state_dict(),
    'swa_model_state_dict': swa_model.state_dict(),
    'scaler_img': scaler_img,
    'scaler_txt': scaler_txt,
    'history': history,
    'config': {
        'dim_img': X_image.shape[1],
        'dim_txt': X_text.shape[1],
        'hidden_dims': [1536, 768, 384],
        'dropout': 0.15,
        'num_heads': 12
    }
}, "/content/drive/MyDrive/advanced_price_model.pt")
print("\n✅ Model saved as 'advanced_price_model.pt'")

NameError: name 'df' is not defined

.

.

.

.

.

**Start of ENSEMBLE LEARNING**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
from tqdm import tqdm
import ast
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
import json

# -------------------------------
# Model Architecture (must match training)
# -------------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        w = torch.mean(x, dim=0, keepdim=True)
        w = torch.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout)
        )
        self.se = SEBlock(dim)
        self.layer_norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.net(x)
        x = self.se(x)
        return self.layer_norm(x + residual)

class GatedCrossAttention(nn.Module):
    def __init__(self, dim_img, dim_txt, dim_hidden, num_heads=8, dropout=0.15):
        super().__init__()
        self.num_heads = num_heads
        self.dim_hidden = dim_hidden
        self.head_dim = dim_hidden // num_heads

        self.query_img = nn.Linear(dim_img, dim_hidden)
        self.key_txt = nn.Linear(dim_txt, dim_hidden)
        self.value_txt = nn.Linear(dim_txt, dim_hidden)
        self.key_img = nn.Linear(dim_img, dim_hidden)
        self.value_img = nn.Linear(dim_img, dim_hidden)

        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Sequential(
            nn.Linear(dim_img + dim_txt, dim_hidden),
            nn.Sigmoid()
        )

        self.out_proj = nn.Linear(dim_hidden * 2, dim_hidden)
        self.layer_norm = nn.LayerNorm(dim_hidden)

    def forward(self, img, txt):
        batch_size = img.size(0)

        Q_img = self.query_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K_txt = self.key_txt(txt).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V_txt = self.value_txt(txt).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        attn_txt = torch.softmax(torch.matmul(Q_img, K_txt.transpose(-2, -1)) * self.scale, dim=-1)
        attn_txt = self.dropout(attn_txt)
        cross_txt = torch.matmul(attn_txt, V_txt).transpose(1, 2).contiguous().view(batch_size, self.dim_hidden)

        K_img = self.key_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V_img = self.value_img(img).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)

        attn_img = torch.softmax(torch.matmul(Q_img, K_img.transpose(-2, -1)) * self.scale, dim=-1)
        attn_img = self.dropout(attn_img)
        self_img = torch.matmul(attn_img, V_img).transpose(1, 2).contiguous().view(batch_size, self.dim_hidden)

        gate_input = torch.cat([img, txt], dim=-1)
        gate_weights = self.gate(gate_input)

        combined = torch.cat([cross_txt, self_img], dim=-1)
        fused = self.out_proj(combined)
        fused = gate_weights * fused

        return self.layer_norm(fused)

class AdvancedPriceModel(nn.Module):
    def __init__(self, dim_img, dim_txt, hidden_dims=[1536, 768, 384], dropout=0.15, num_heads=12):
        super().__init__()

        self.img_proj = nn.Sequential(
            nn.Linear(dim_img, hidden_dims[0] // 2),
            nn.LayerNorm(hidden_dims[0] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )

        self.txt_proj = nn.Sequential(
            nn.Linear(dim_txt, hidden_dims[0] // 2),
            nn.LayerNorm(hidden_dims[0] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )

        self.cross_attn = GatedCrossAttention(
            hidden_dims[0] // 2,
            hidden_dims[0] // 2,
            hidden_dims[0],
            num_heads=num_heads,
            dropout=dropout
        )

        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_dims[0], dropout) for _ in range(3)
        ])

        self.down_blocks = nn.ModuleList()
        prev_dim = hidden_dims[0]
        for h_dim in hidden_dims[1:]:
            self.down_blocks.append(nn.Sequential(
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ))
            prev_dim = h_dim

        self.skip_connections = nn.ModuleList([
            nn.Linear(hidden_dims[0], hidden_dims[-1])
        ])

        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dims[-1] * 2, hidden_dims[-1]),
            nn.LayerNorm(hidden_dims[-1]),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dims[-1], hidden_dims[-1] // 2),
            nn.LayerNorm(hidden_dims[-1] // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.25),
            nn.Linear(hidden_dims[-1] // 2, 1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, img, txt):
        img_feat = self.img_proj(img)
        txt_feat = self.txt_proj(txt)

        x = self.cross_attn(img_feat, txt_feat)
        skip = x

        for block in self.residual_blocks:
            x = block(x)

        for down in self.down_blocks:
            x = down(x)

        skip_transformed = self.skip_connections[0](skip)
        x = torch.cat([x, skip_transformed], dim=-1)

        out = self.prediction_head(x)
        return out.squeeze()

In [ ]:
# ============================================================
# MULTIMODAL ENSEMBLE: TRAIN, SAVE, AND INFER ON TEST
# Produces /content/drive/MyDrive/Asad/ensemble_final_predictions.csv
# Saves models & scalers in /content/drive/MyDrive/Asad/ensemble_artifacts/
# ============================================================
import os, json, math, ast, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV, Ridge
from joblib import dump, load
from tqdm import tqdm

# -------------------------------
# Config / Paths
# -------------------------------
BASE_DIR = "/content/drive/MyDrive/"
TRAIN_CSV = f"{BASE_DIR}/Asad/train_combined.csv"
TEST_CSV  = f"{BASE_DIR}/Asad/test_combined.csv"
PRETRAINED_PT = f"{BASE_DIR}/advanced_price_model.pt"
ARTIFACT_DIR = f"{BASE_DIR}/ensemble_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------------------------------
# Metric
# -------------------------------
def msape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps
    return 100.0 * np.mean(np.abs(y_true - y_pred) / denom)

# -------------------------------
# Helper function to safely parse encodings
# -------------------------------
def safe_parse(x):
    if pd.isna(x):
        return None
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None
    if isinstance(x, (list, np.ndarray)):
        return x
    return None

# -------------------------------
# Pretrained multimodal model (your architecture)
# -------------------------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        w = torch.mean(x, dim=0, keepdim=True)
        w = torch.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout)
        )
        self.se = SEBlock(dim)
        self.layer_norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.net(x)
        x = self.se(x)
        return self.layer_norm(x + residual)

class GatedCrossAttention(nn.Module):
    def __init__(self, dim_img, dim_txt, dim_hidden, num_heads=8, dropout=0.15):
        super().__init__()
        self.num_heads = num_heads
        self.dim_hidden = dim_hidden
        self.head_dim = dim_hidden // num_heads

        self.query_img = nn.Linear(dim_img, dim_hidden)
        self.key_txt = nn.Linear(dim_txt, dim_hidden)
        self.value_txt = nn.Linear(dim_txt, dim_hidden)
        self.key_img = nn.Linear(dim_img, dim_hidden)
        self.value_img = nn.Linear(dim_img, dim_hidden)

        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Sequential(nn.Linear(dim_img + dim_txt, dim_hidden), nn.Sigmoid())
        self.out_proj = nn.Linear(dim_hidden * 2, dim_hidden)
        self.layer_norm = nn.LayerNorm(dim_hidden)

    def forward(self, img, txt):
        B = img.size(0)
        Q = self.query_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Kt = self.key_txt(txt).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Vt = self.value_txt(txt).view(B,1,self.num_heads,self.head_dim).transpose(1,2)

        att_t = torch.softmax(torch.matmul(Q, Kt.transpose(-2,-1))*self.scale, dim=-1)
        cross_t = torch.matmul(att_t, Vt).transpose(1,2).reshape(B, self.dim_hidden)

        Ki = self.key_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Vi = self.value_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)

        att_i = torch.softmax(torch.matmul(Q, Ki.transpose(-2,-1))*self.scale, dim=-1)
        self_i = torch.matmul(att_i, Vi).transpose(1,2).reshape(B, self.dim_hidden)

        gate = self.gate(torch.cat([img, txt], dim=-1))
        fused = self.out_proj(torch.cat([cross_t, self_i], dim=-1))
        return self.layer_norm(gate * fused)

class AdvancedPriceModel(nn.Module):
    def __init__(self, dim_img, dim_txt, hidden_dims=[1536,768,384], dropout=0.15, num_heads=12):
        super().__init__()
        self.img_proj = nn.Sequential(
            nn.Linear(dim_img, hidden_dims[0]//2), nn.LayerNorm(hidden_dims[0]//2),
            nn.GELU(), nn.Dropout(dropout*0.5)
        )
        self.txt_proj = nn.Sequential(
            nn.Linear(dim_txt, hidden_dims[0]//2), nn.LayerNorm(hidden_dims[0]//2),
            nn.GELU(), nn.Dropout(dropout*0.5)
        )
        self.cross_attn = GatedCrossAttention(hidden_dims[0]//2, hidden_dims[0]//2, hidden_dims[0],
                                              num_heads=num_heads, dropout=dropout)
        self.res_blocks = nn.ModuleList([ResidualBlock(hidden_dims[0], dropout) for _ in range(3)])
        self.down_blocks = nn.ModuleList()
        prev = hidden_dims[0]
        for h in hidden_dims[1:]:
            self.down_blocks.append(nn.Sequential(
                nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)
            ))
            prev = h
        self.skip = nn.Linear(hidden_dims[0], hidden_dims[-1])
        self.pred = nn.Sequential(
            nn.Linear(hidden_dims[-1]*2, hidden_dims[-1]), nn.LayerNorm(hidden_dims[-1]),
            nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(hidden_dims[-1], hidden_dims[-1]//2), nn.LayerNorm(hidden_dims[-1]//2),
            nn.GELU(), nn.Dropout(dropout*0.25),
            nn.Linear(hidden_dims[-1]//2, 1)
        )

    def forward(self, img, txt):
        i = self.img_proj(img)
        t = self.txt_proj(txt)
        x = self.cross_attn(i, t)
        skip = x
        for blk in self.res_blocks:
            x = blk(x)
        for d in self.down_blocks:
            x = d(x)
        x = torch.cat([x, self.skip(skip)], dim=-1)
        return self.pred(x).squeeze(-1)

# -------------------------------
# Function to rename state dict keys to match current model
# -------------------------------
def rename_state_dict_keys(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        if 'residual_blocks' in key:
            new_key = key.replace('residual_blocks', 'res_blocks')
        elif 'skip_connections.0' in key:
            new_key = key.replace('skip_connections.0', 'skip')
        elif 'prediction_head' in key:
            new_key = key.replace('prediction_head', 'pred')
        new_state_dict[new_key] = value
    return new_state_dict

# -------------------------------
# Unimodal MLP (log-price)
# -------------------------------
class UniMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 1024), nn.LayerNorm(1024), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_unimodal(model, Xtr, ylog_tr, Xva, ylog_va, epochs=60, lr=1e-3, wd=1e-4):
    tr = TensorDataset(torch.tensor(Xtr).float(), torch.tensor(ylog_tr).float())
    va = TensorDataset(torch.tensor(Xva).float(), torch.tensor(ylog_va).float())
    tl = DataLoader(tr, batch_size=256, shuffle=True)
    vl = DataLoader(va, batch_size=512, shuffle=False)

    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    best, best_s = float("inf"), None
    patience = 10

    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        vals = []
        with torch.no_grad():
            for xb, yb in vl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vals.append(loss_fn(model(xb), yb).item())

        vm = float(np.mean(vals))
        if vm < best - 1e-6:
            best, best_s = vm, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience = 10
        else:
            patience -= 1
            if patience <= 0:
                break

    if best_s:
        model.load_state_dict(best_s)
    return model

@torch.no_grad()
def predict_uni_log(model, X):
    model.eval()
    outs = []
    X_t = torch.tensor(X).float()
    for i in range(0, len(X), 512):
        outs.append(model(X_t[i:i+512].to(DEVICE)).cpu().numpy())
    return np.concatenate(outs, axis=0)

@torch.no_grad()
def predict_multi_log(model, Xi, Xt):
    model.eval()
    outs = []
    Xi_t = torch.tensor(Xi).float()
    Xt_t = torch.tensor(Xt).float()
    for i in range(0, len(Xi), 512):
        outs.append(model(Xi_t[i:i+512].to(DEVICE), Xt_t[i:i+512].to(DEVICE)).cpu().numpy())
    return np.concatenate(outs, axis=0)

# -------------------------------
# Load training data
# -------------------------------
print("Loading training data...")
df = pd.read_csv(TRAIN_CSV).dropna()

has_sample_id = 'sample_id' in df.columns
if has_sample_id:
    tr_sample_ids = df['sample_id'].values

drop_cols = [c for c in ['sample_id','catalog_content','image_link','download_failed'] if c in df.columns]
df = df.drop(columns=drop_cols)

df['image_encoding'] = df['image_encoding'].apply(safe_parse)
df['text_encoding'] = df['text_encoding'].apply(safe_parse)

X_img = np.asarray(df['image_encoding'].tolist(), dtype=np.float32)
X_txt = np.asarray(df['text_encoding'].tolist(), dtype=np.float32)
y = df['price'].values.astype(np.float32)
y_log = np.log1p(y)

# -------------------------------
# Load pretrained multimodal & feature scalers
# -------------------------------
print("Loading pretrained model...")
ckpt = torch.load(PRETRAINED_PT, map_location='cpu', weights_only=False)
config = ckpt['config']
scaler_img = ckpt['scaler_img']
scaler_txt = ckpt['scaler_txt']
target_is_log = bool(config.get('target_is_log', True))

Xi_s = scaler_img.transform(X_img)
Xt_s = scaler_txt.transform(X_txt)

multi = AdvancedPriceModel(
    dim_img=config['dim_img'],
    dim_txt=config['dim_txt'],
    hidden_dims=config['hidden_dims'],
    dropout=config['dropout'],
    num_heads=config['num_heads']
).to(DEVICE)

original_state_dict = ckpt['model_state_dict']
renamed_state_dict = rename_state_dict_keys(original_state_dict)
multi.load_state_dict(renamed_state_dict)
multi.eval()
print("Pretrained model loaded successfully!")

# -------------------------------
# Train/val split
# -------------------------------
Xi_tr, Xi_va, Xt_tr, Xt_va, ylog_tr, ylog_va = train_test_split(
    Xi_s, Xt_s, y_log, test_size=0.1, random_state=SEED, shuffle=True
)

# -------------------------------
# OOF predictions for stacking
# -------------------------------
print("\nGenerating OOF predictions...")
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_img, oof_txt, oof_mul, oof_y = [], [], [], []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(Xi_tr)):
    print(f"  Fold {fold_idx+1}/5...")
    Xi_t, Xi_v = Xi_tr[tr_idx], Xi_tr[va_idx]
    Xt_t, Xt_v = Xt_tr[tr_idx], Xt_tr[va_idx]
    ylt, ylv = ylog_tr[tr_idx], ylog_tr[va_idx]

    img_m = train_unimodal(UniMLP(Xi_t.shape[1]), Xi_t, ylt, Xi_v, ylv, epochs=50)
    txt_m = train_unimodal(UniMLP(Xt_t.shape[1]), Xt_t, ylt, Xt_v, ylv, epochs=50)

    img_o = np.expm1(predict_uni_log(img_m, Xi_v))
    txt_o = np.expm1(predict_uni_log(txt_m, Xt_v))
    mul_o = np.expm1(predict_multi_log(multi, Xi_v, Xt_v)) if target_is_log else predict_multi_log(multi, Xi_v, Xt_v)

    oof_img.append(img_o)
    oof_txt.append(txt_o)
    oof_mul.append(mul_o)
    oof_y.append(np.expm1(ylv))

img_oof = np.concatenate(oof_img)
txt_oof = np.concatenate(oof_txt)
mul_oof = np.concatenate(oof_mul)
y_oof = np.concatenate(oof_y)

# -------------------------------
# Meta features + scaler
# -------------------------------
print("\nCreating meta features...")
meta_X = np.column_stack([
    img_oof, txt_oof, mul_oof,
    np.abs(img_oof - mul_oof),
    np.abs(txt_oof - mul_oof),
    np.log1p(mul_oof)
])

meta_scaler = RobustScaler()
meta_X = meta_scaler.fit_transform(meta_X)

# -------------------------------
# RidgeCV meta learner
# -------------------------------
print("Training Ridge ensemble...")
alphas = np.logspace(-3, 2, 15)
ridge_cv = RidgeCV(alphas=alphas, fit_intercept=True, scoring='neg_mean_absolute_error')
ridge_cv.fit(meta_X, y_oof)
alpha_best = ridge_cv.alpha_

ridge = Ridge(alpha=alpha_best, fit_intercept=True, positive=True)
ridge.fit(meta_X, y_oof)
ridge_oof = ridge.predict(meta_X)
print(f"OOF Ridge ensemble MSAPE: {msape(y_oof, ridge_oof):.3f}")

# -------------------------------
# Meta-MLP learner
# -------------------------------
print("Training Meta-MLP ensemble...")
meta_mlp = nn.Sequential(
    nn.Linear(meta_X.shape[1], 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(32, 1)
).to(DEVICE)

opt = torch.optim.AdamW(meta_mlp.parameters(), lr=5e-4, weight_decay=1e-3)
crit = nn.SmoothL1Loss()
Xt = torch.tensor(meta_X, dtype=torch.float32, device=DEVICE)
yt = torch.tensor(y_oof, dtype=torch.float32, device=DEVICE)

best_loss, best_state = float("inf"), None
for _ in tqdm(range(400), desc="Train Meta-MLP"):
    opt.zero_grad()
    out = meta_mlp(Xt).squeeze(-1)
    loss = crit(out, yt)
    loss.backward()
    opt.step()
    if loss.item() < best_loss - 1e-6:
        best_loss = loss.item()
        best_state = {k:v.detach().cpu().clone() for k,v in meta_mlp.state_dict().items()}

if best_state:
    meta_mlp.load_state_dict(best_state)

mlp_oof = meta_mlp(Xt).detach().cpu().numpy().squeeze()
print(f"OOF Meta-MLP MSAPE: {msape(y_oof, mlp_oof):.3f}")

# -------------------------------
# Evaluate on validation split
# -------------------------------
print("\nEvaluating on validation split...")
img_full = train_unimodal(UniMLP(Xi_s.shape[1]), Xi_s, y_log, Xi_va, ylog_va, epochs=150)
txt_full = train_unimodal(UniMLP(Xt_s.shape[1]), Xt_s, y_log, Xt_va, ylog_va, epochs=60)

img_va = np.expm1(predict_uni_log(img_full, Xi_va))
txt_va = np.expm1(predict_uni_log(txt_full, Xt_va))
mul_va = np.expm1(predict_multi_log(multi, Xi_va, Xt_va)) if target_is_log else predict_multi_log(multi, Xi_va, Xt_va)
y_val = np.expm1(ylog_va)

meta_val = np.column_stack([
    img_va, txt_va, mul_va,
    np.abs(img_va - mul_va),
    np.abs(txt_va - mul_va),
    np.log1p(mul_va)
])
meta_val_s = meta_scaler.transform(meta_val)

ridge_val = ridge.predict(meta_val_s)
mlp_val = meta_mlp(torch.tensor(meta_val_s, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy().squeeze()

ms_ridge = msape(y_val, ridge_val)
ms_mlp = msape(y_val, mlp_val)

print("\nHoldout validation MSAPE:")
print(f"  Ridge ensemble : {ms_ridge:.3f}")
print(f"  MLP ensemble   : {ms_mlp:.3f}")

best_name = "ridge" if ms_ridge <= ms_mlp else "mlp"
print(f"\nSelected ensemble: {best_name.upper()}")

# -------------------------------
# SAVE ARTIFACTS
# -------------------------------
print(f"\nSaving artifacts to {ARTIFACT_DIR}...")
torch.save(img_full.state_dict(), f"{ARTIFACT_DIR}/image_only_model.pt")
torch.save(txt_full.state_dict(), f"{ARTIFACT_DIR}/text_only_model.pt")
dump(meta_scaler, f"{ARTIFACT_DIR}/meta_scaler.joblib")
dump(ridge, f"{ARTIFACT_DIR}/ridge_model.joblib")
torch.save(meta_mlp.state_dict(), f"{ARTIFACT_DIR}/meta_mlp.pt")

save_config = {
    "best_ensemble": best_name,
    "alpha_best": float(alpha_best),
    "target_is_log": bool(target_is_log),
    "img_dim": int(Xi_s.shape[1]),
    "txt_dim": int(Xt_s.shape[1]),
    "paths": {
        "img_model": f"{ARTIFACT_DIR}/image_only_model.pt",
        "txt_model": f"{ARTIFACT_DIR}/text_only_model.pt",
        "meta_scaler": f"{ARTIFACT_DIR}/meta_scaler.joblib",
        "ridge": f"{ARTIFACT_DIR}/ridge_model.joblib",
        "meta_mlp": f"{ARTIFACT_DIR}/meta_mlp.pt",
        "pretrained_pt": PRETRAINED_PT
    }
}

with open(f"{ARTIFACT_DIR}/ensemble_config.json", "w") as f:
    json.dump(save_config, f, indent=2)

print("Artifacts saved successfully!")

# ============================================================
# INFERENCE ON TEST SET
# ============================================================
print("\n" + "="*60)
print("Starting inference on test set...")

test_df = pd.read_csv(TEST_CSV)
test_sample_ids = test_df['sample_id'].values if 'sample_id' in test_df.columns else np.arange(len(test_df))

# Parse encodings with safe_parse
print("Parsing test encodings...")
test_df['image_encoding'] = test_df['image_encoding'].apply(safe_parse)
test_df['text_encoding'] = test_df['text_encoding'].apply(safe_parse)

# Check for missing values
missing_img = test_df['image_encoding'].isna().sum()
missing_txt = test_df['text_encoding'].isna().sum()
print(f"Missing image encodings: {missing_img}")
print(f"Missing text encodings: {missing_txt}")

# Get dimensions from first valid encoding
first_valid_img = test_df['image_encoding'].dropna().iloc[0]
first_valid_txt = test_df['text_encoding'].dropna().iloc[0]
img_dim = len(first_valid_img)
txt_dim = len(first_valid_txt)

# Fill missing with zeros
def fill_missing(x, dim):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.zeros(dim)
    return x

test_df['image_encoding'] = test_df['image_encoding'].apply(lambda x: fill_missing(x, img_dim))
test_df['text_encoding'] = test_df['text_encoding'].apply(lambda x: fill_missing(x, txt_dim))

# Convert to arrays
Xi_te = np.asarray(test_df['image_encoding'].tolist(), dtype=np.float32)
Xt_te = np.asarray(test_df['text_encoding'].tolist(), dtype=np.float32)

print(f"Test image shape: {Xi_te.shape}")
print(f"Test text shape: {Xt_te.shape}")

# Scale with pretrained scalers
Xi_te_s = scaler_img.transform(Xi_te)
Xt_te_s = scaler_txt.transform(Xt_te)

# Generate predictions
print("Generating test predictions...")
img_pred_log = predict_uni_log(img_full, Xi_te_s)
txt_pred_log = predict_uni_log(txt_full, Xt_te_s)
mul_pred_log = predict_multi_log(multi, Xi_te_s, Xt_te_s)

img_p = np.expm1(img_pred_log)
txt_p = np.expm1(txt_pred_log)
mul_p = np.expm1(mul_pred_log) if target_is_log else mul_pred_log

# Create meta features
meta_test = np.column_stack([
    img_p, txt_p, mul_p,
    np.abs(img_p - mul_p),
    np.abs(txt_p - mul_p),
    np.log1p(mul_p)
])
meta_test_s = meta_scaler.transform(meta_test)

# Generate final ensemble prediction
if best_name == "ridge":
    final_preds = ridge.predict(meta_test_s)
else:
    final_preds = meta_mlp(torch.tensor(meta_test_s, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy().squeeze()

# Ensure non-negative prices
final_preds = np.maximum(final_preds, 0.0)

# Save predictions
out_df = pd.DataFrame({
    "sample_id": test_sample_ids,
    "price": final_preds
})

OUT_CSV = f"{BASE_DIR}/ensemble_final_predictions2.csv"
out_df.to_csv(OUT_CSV, index=False)

print(f"\n✅ Predictions saved to: {OUT_CSV}")
print(f"Total test samples: {len(out_df)}")
print(f"Price range: ${final_preds.min():.2f} - ${final_preds.max():.2f}")
print(f"Mean price: ${final_preds.mean():.2f}")
print(f"Median price: ${np.median(final_preds):.2f}")
print("="*60)
print("\nFirst 10 predictions:")
print(out_df.head(10))

Loading training data...


In [ ]:
# ============================================================
# OPTIMIZED MULTIMODAL ENSEMBLE: Enhanced for Better MSAPE
# Key improvements:
# 1. Better meta-feature engineering (confidence scores, ratios)
# 2. Quantile regression for robustness
# 3. Stacking with LightGBM
# 4. Advanced regularization & augmentation
# 5. Voting ensemble with learned weights
# ============================================================
import os, json, math, ast, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import Ridge, QuantileRegressor
from sklearn.ensemble import GradientBoostingRegressor
from joblib import dump, load
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Try to import LightGBM (optional but recommended)
try:
    import lightgbm as lgb
    HAS_LGB = True
except:
    HAS_LGB = False

# -------------------------------
# Config / Paths
# -------------------------------
BASE_DIR = "/content/drive/MyDrive/"
TRAIN_CSV = f"{BASE_DIR}/Asad/train_combined.csv"
TEST_CSV  = f"{BASE_DIR}/Asad/test_combined.csv"
PRETRAINED_PT = f"{BASE_DIR}/advanced_price_model.pt"
ARTIFACT_DIR = f"{BASE_DIR}/ensemble_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Enhanced configuration
CONFIG = {
    'epochs_uni': 100,  # Increased from 50-60
    'epochs_meta': 600,  # Increased from 400
    'lr_uni': 1e-3,
    'wd_uni': 1e-4,
    'lr_meta': 3e-4,  # Decreased for stability
    'wd_meta': 5e-4,  # Increased regularization
    'batch_size': 256,
    'meta_dropout': 0.4,  # Increased dropout
    'patience': 15,  # Increased patience
}

# -------------------------------
# Improved Metric
# -------------------------------
def msape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps
    return 100.0 * np.mean(np.abs(y_true - y_pred) / denom)

def mape(y_true, y_pred, eps=1e-9):
    return 100.0 * np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps)))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# -------------------------------
# Helper function
# -------------------------------
def safe_parse(x):
    if pd.isna(x):
        return None
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            try:
                return ast.literal_eval(x)
            except:
                return None
    if isinstance(x, (list, np.ndarray)):
        return x
    return None

# ============= NEURAL NETWORK COMPONENTS =============

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, max(1, channels // reduction))
        self.fc2 = nn.Linear(max(1, channels // reduction), channels)

    def forward(self, x):
        w = torch.mean(x, dim=0, keepdim=True)
        w = torch.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout)
        )
        self.se = SEBlock(dim)
        self.layer_norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.net(x)
        x = self.se(x)
        return self.layer_norm(x + residual)

class GatedCrossAttention(nn.Module):
    def __init__(self, dim_img, dim_txt, dim_hidden, num_heads=8, dropout=0.15):
        super().__init__()
        self.num_heads = num_heads
        self.dim_hidden = dim_hidden
        self.head_dim = dim_hidden // num_heads

        self.query_img = nn.Linear(dim_img, dim_hidden)
        self.key_txt = nn.Linear(dim_txt, dim_hidden)
        self.value_txt = nn.Linear(dim_txt, dim_hidden)
        self.key_img = nn.Linear(dim_img, dim_hidden)
        self.value_img = nn.Linear(dim_img, dim_hidden)

        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Sequential(nn.Linear(dim_img + dim_txt, dim_hidden), nn.Sigmoid())
        self.out_proj = nn.Linear(dim_hidden * 2, dim_hidden)
        self.layer_norm = nn.LayerNorm(dim_hidden)

    def forward(self, img, txt):
        B = img.size(0)
        Q = self.query_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Kt = self.key_txt(txt).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Vt = self.value_txt(txt).view(B,1,self.num_heads,self.head_dim).transpose(1,2)

        att_t = torch.softmax(torch.matmul(Q, Kt.transpose(-2,-1))*self.scale, dim=-1)
        cross_t = torch.matmul(att_t, Vt).transpose(1,2).reshape(B, self.dim_hidden)

        Ki = self.key_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)
        Vi = self.value_img(img).view(B,1,self.num_heads,self.head_dim).transpose(1,2)

        att_i = torch.softmax(torch.matmul(Q, Ki.transpose(-2,-1))*self.scale, dim=-1)
        self_i = torch.matmul(att_i, Vi).transpose(1,2).reshape(B, self.dim_hidden)

        gate = self.gate(torch.cat([img, txt], dim=-1))
        fused = self.out_proj(torch.cat([cross_t, self_i], dim=-1))
        return self.layer_norm(gate * fused)

class AdvancedPriceModel(nn.Module):
    def __init__(self, dim_img, dim_txt, hidden_dims=[1536,768,384], dropout=0.15, num_heads=12):
        super().__init__()
        self.img_proj = nn.Sequential(
            nn.Linear(dim_img, hidden_dims[0]//2), nn.LayerNorm(hidden_dims[0]//2),
            nn.GELU(), nn.Dropout(dropout*0.5)
        )
        self.txt_proj = nn.Sequential(
            nn.Linear(dim_txt, hidden_dims[0]//2), nn.LayerNorm(hidden_dims[0]//2),
            nn.GELU(), nn.Dropout(dropout*0.5)
        )
        self.cross_attn = GatedCrossAttention(hidden_dims[0]//2, hidden_dims[0]//2, hidden_dims[0],
                                              num_heads=num_heads, dropout=dropout)
        self.res_blocks = nn.ModuleList([ResidualBlock(hidden_dims[0], dropout) for _ in range(3)])
        self.down_blocks = nn.ModuleList()
        prev = hidden_dims[0]
        for h in hidden_dims[1:]:
            self.down_blocks.append(nn.Sequential(
                nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)
            ))
            prev = h
        self.skip = nn.Linear(hidden_dims[0], hidden_dims[-1])
        self.pred = nn.Sequential(
            nn.Linear(hidden_dims[-1]*2, hidden_dims[-1]), nn.LayerNorm(hidden_dims[-1]),
            nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(hidden_dims[-1], hidden_dims[-1]//2), nn.LayerNorm(hidden_dims[-1]//2),
            nn.GELU(), nn.Dropout(dropout*0.25),
            nn.Linear(hidden_dims[-1]//2, 1)
        )

    def forward(self, img, txt):
        i = self.img_proj(img)
        t = self.txt_proj(txt)
        x = self.cross_attn(i, t)
        skip = x
        for blk in self.res_blocks:
            x = blk(x)
        for d in self.down_blocks:
            x = d(x)
        x = torch.cat([x, self.skip(skip)], dim=-1)
        return self.pred(x).squeeze(-1)

def rename_state_dict_keys(state_dict):
    new_state_dict = {}
    for key, value in state_dict.items():
        new_key = key
        if 'residual_blocks' in key:
            new_key = key.replace('residual_blocks', 'res_blocks')
        elif 'skip_connections.0' in key:
            new_key = key.replace('skip_connections.0', 'skip')
        elif 'prediction_head' in key:
            new_key = key.replace('prediction_head', 'pred')
        new_state_dict[new_key] = value
    return new_state_dict

# ============= IMPROVED UNIMODAL MLP =============

class ImprovedUniMLP(nn.Module):
    def __init__(self, dim, hidden=[1024, 512, 256]):
        super().__init__()
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(0.2)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_unimodal(model, Xtr, ylog_tr, Xva, ylog_va, epochs=100, lr=1e-3, wd=1e-4):
    tr = TensorDataset(torch.tensor(Xtr).float(), torch.tensor(ylog_tr).float())
    va = TensorDataset(torch.tensor(Xva).float(), torch.tensor(ylog_va).float())
    tl = DataLoader(tr, batch_size=CONFIG['batch_size'], shuffle=True)
    vl = DataLoader(va, batch_size=512, shuffle=False)

    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    best, best_s = float("inf"), None
    patience = CONFIG['patience']
    patience_cnt = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vals = []
        with torch.no_grad():
            for xb, yb in vl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vals.append(loss_fn(model(xb), yb).item())

        vm = float(np.mean(vals))
        if vm < best - 1e-6:
            best, best_s = vm, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                break

    if best_s:
        model.load_state_dict(best_s)
    return model

@torch.no_grad()
def predict_uni_log(model, X):
    model.eval()
    outs = []
    X_t = torch.tensor(X).float()
    for i in range(0, len(X), 512):
        outs.append(model(X_t[i:i+512].to(DEVICE)).cpu().numpy())
    return np.concatenate(outs, axis=0)

@torch.no_grad()
def predict_multi_log(model, Xi, Xt):
    model.eval()
    outs = []
    Xi_t = torch.tensor(Xi).float()
    Xt_t = torch.tensor(Xt).float()
    for i in range(0, len(Xi), 512):
        outs.append(model(Xi_t[i:i+512].to(DEVICE), Xt_t[i:i+512].to(DEVICE)).cpu().numpy())
    return np.concatenate(outs, axis=0)

# ============= ENHANCED META-FEATURES =============

def create_advanced_meta_features(img_pred, txt_pred, mul_pred):
    """Enhanced meta-features with confidence & interaction terms"""
    meta = [img_pred, txt_pred, mul_pred]

    # Absolute differences (disagreement)
    meta.append(np.abs(img_pred - mul_pred))
    meta.append(np.abs(txt_pred - mul_pred))
    meta.append(np.abs(img_pred - txt_pred))

    # Log-transformed predictions (for scale-invariance)
    meta.append(np.log1p(mul_pred))
    meta.append(np.log1p(np.abs(img_pred)))
    meta.append(np.log1p(np.abs(txt_pred)))

    # Ratios (normalized)
    safe_mul = np.maximum(np.abs(mul_pred), 1e-6)
    meta.append(img_pred / safe_mul)
    meta.append(txt_pred / safe_mul)

    # Median of predictions
    all_preds = np.stack([img_pred, txt_pred, mul_pred], axis=1)
    meta.append(np.median(all_preds, axis=1))

    # Variance of predictions (disagreement)
    meta.append(np.var(all_preds, axis=1))

    # Min/Max
    meta.append(np.min(all_preds, axis=1))
    meta.append(np.max(all_preds, axis=1))

    # Range
    meta.append(np.max(all_preds, axis=1) - np.min(all_preds, axis=1))

    return np.column_stack(meta)

# ============= ENHANCED META-MLP =============

class EnhancedMetaMLP(nn.Module):
    def __init__(self, input_dim, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.8),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout * 0.6),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

# ============= LOAD DATA =============

print("Loading training data...")
df = pd.read_csv(TRAIN_CSV).dropna()

has_sample_id = 'sample_id' in df.columns
if has_sample_id:
    tr_sample_ids = df['sample_id'].values

drop_cols = [c for c in ['sample_id','catalog_content','image_link','download_failed'] if c in df.columns]
df = df.drop(columns=drop_cols)

df['image_encoding'] = df['image_encoding'].apply(safe_parse)
df['text_encoding'] = df['text_encoding'].apply(safe_parse)

X_img = np.asarray(df['image_encoding'].tolist(), dtype=np.float32)
X_txt = np.asarray(df['text_encoding'].tolist(), dtype=np.float32)
y = df['price'].values.astype(np.float32)
y_log = np.log1p(y)

# ============= LOAD PRETRAINED =============

print("Loading pretrained model...")
ckpt = torch.load(PRETRAINED_PT, map_location='cpu', weights_only=False)
config = ckpt['config']
scaler_img = ckpt['scaler_img']
scaler_txt = ckpt['scaler_txt']
target_is_log = bool(config.get('target_is_log', True))

Xi_s = scaler_img.transform(X_img)
Xt_s = scaler_txt.transform(X_txt)

multi = AdvancedPriceModel(
    dim_img=config['dim_img'],
    dim_txt=config['dim_txt'],
    hidden_dims=config['hidden_dims'],
    dropout=config['dropout'],
    num_heads=config['num_heads']
).to(DEVICE)

original_state_dict = ckpt['model_state_dict']
renamed_state_dict = rename_state_dict_keys(original_state_dict)
multi.load_state_dict(renamed_state_dict)
multi.eval()
print("Pretrained model loaded!")

# ============= TRAIN/VAL SPLIT =============

Xi_tr, Xi_va, Xt_tr, Xt_va, ylog_tr, ylog_va = train_test_split(
    Xi_s, Xt_s, y_log, test_size=0.15, random_state=SEED, shuffle=True
)

# ============= OOF PREDICTIONS WITH KFOLD =============

print("\nGenerating OOF predictions (5-fold)...")
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_img, oof_txt, oof_mul, oof_y = [], [], [], []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(Xi_tr)):
    print(f"  Fold {fold_idx+1}/5...")
    Xi_t, Xi_v = Xi_tr[tr_idx], Xi_tr[va_idx]
    Xt_t, Xt_v = Xt_tr[tr_idx], Xt_tr[va_idx]
    ylt, ylv = ylog_tr[tr_idx], ylog_tr[va_idx]

    img_m = train_unimodal(
        ImprovedUniMLP(Xi_t.shape[1]),
        Xi_t, ylt, Xi_v, ylv,
        epochs=CONFIG['epochs_uni'], lr=CONFIG['lr_uni'], wd=CONFIG['wd_uni']
    )
    txt_m = train_unimodal(
        ImprovedUniMLP(Xt_t.shape[1]),
        Xt_t, ylt, Xt_v, ylv,
        epochs=CONFIG['epochs_uni'], lr=CONFIG['lr_uni'], wd=CONFIG['wd_uni']
    )

    img_o = np.expm1(predict_uni_log(img_m, Xi_v))
    txt_o = np.expm1(predict_uni_log(txt_m, Xt_v))
    mul_o = np.expm1(predict_multi_log(multi, Xi_v, Xt_v)) if target_is_log else predict_multi_log(multi, Xi_v, Xt_v)

    oof_img.append(img_o)
    oof_txt.append(txt_o)
    oof_mul.append(mul_o)
    oof_y.append(np.expm1(ylv))

img_oof = np.concatenate(oof_img)
txt_oof = np.concatenate(oof_txt)
mul_oof = np.concatenate(oof_mul)
y_oof = np.concatenate(oof_y)

# ============= ADVANCED META-FEATURES =============

print("\nCreating advanced meta features...")
meta_X = create_advanced_meta_features(img_oof, txt_oof, mul_oof)

meta_scaler = RobustScaler()
meta_X_scaled = meta_scaler.fit_transform(meta_X)

print(f"Meta features shape: {meta_X_scaled.shape}")

# ============= RIDGE REGRESSION =============

print("Training Ridge ensemble...")
alphas = np.logspace(-2, 3, 20)
ridge_cv = Ridge(alpha=100.0, fit_intercept=True)  # Pre-tuned alpha
ridge_cv.fit(meta_X_scaled, y_oof)
ridge_oof = ridge_cv.predict(meta_X_scaled)
ridge_msape = msape(y_oof, ridge_oof)
print(f"OOF Ridge MSAPE: {ridge_msape:.3f}")

# ============= GRADIENT BOOSTING =============

print("Training GradientBoosting ensemble...")
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=SEED,
    validation_fraction=0.1,
    n_iter_no_change=50
)
gb.fit(meta_X_scaled, y_oof)
gb_oof = gb.predict(meta_X_scaled)
gb_msape = msape(y_oof, gb_oof)
print(f"OOF GradientBoosting MSAPE: {gb_msape:.3f}")

# ============= ENHANCED META-MLP =============

print("Training Enhanced Meta-MLP...")
meta_mlp = EnhancedMetaMLP(meta_X_scaled.shape[1], dropout=CONFIG['meta_dropout']).to(DEVICE)

opt = torch.optim.AdamW(meta_mlp.parameters(), lr=CONFIG['lr_meta'], weight_decay=CONFIG['wd_meta'])
crit = nn.SmoothL1Loss()
Xt = torch.tensor(meta_X_scaled, dtype=torch.float32, device=DEVICE)
yt = torch.tensor(y_oof, dtype=torch.float32, device=DEVICE)

best_loss, best_state = float("inf"), None
patience_cnt = 0

for ep in tqdm(range(CONFIG['epochs_meta']), desc="Train Meta-MLP"):
    opt.zero_grad()
    out = meta_mlp(Xt).squeeze(-1)
    loss = crit(out, yt)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(meta_mlp.parameters(), 1.0)
    opt.step()

    if loss.item() < best_loss - 1e-7:
        best_loss = loss.item()
        best_state = {k:v.detach().cpu().clone() for k,v in meta_mlp.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt > 100:
            break

if best_state:
    meta_mlp.load_state_dict(best_state)

mlp_oof = meta_mlp(Xt).detach().cpu().numpy().squeeze()
mlp_msape = msape(y_oof, mlp_oof)
print(f"OOF Meta-MLP MSAPE: {mlp_msape:.3f}")

# ============= VALIDATION =============

print("\nEvaluating on validation split...")
img_full = train_unimodal(
    ImprovedUniMLP(Xi_s.shape[1]),
    Xi_s, y_log, Xi_va, ylog_va,
    epochs=CONFIG['epochs_uni'], lr=CONFIG['lr_uni'], wd=CONFIG['wd_uni']
)
txt_full = train_unimodal(
    ImprovedUniMLP(Xt_s.shape[1]),
    Xt_s, y_log, Xt_va, ylog_va,
    epochs=CONFIG['epochs_uni'], lr=CONFIG['lr_uni'], wd=CONFIG['wd_uni']
)

img_va = np.expm1(predict_uni_log(img_full, Xi_va))
txt_va = np.expm1(predict_uni_log(txt_full, Xt_va))
mul_va = np.expm1(predict_multi_log(multi, Xi_va, Xt_va)) if target_is_log else predict_multi_log(multi, Xi_va, Xt_va)
y_val = np.expm1(ylog_va)

meta_val = create_advanced_meta_features(img_va, txt_va, mul_va)
meta_val_s = meta_scaler.transform(meta_val)

ridge_val = ridge_cv.predict(meta_val_s)
gb_val = gb.predict(meta_val_s)
mlp_val = meta_mlp(torch.tensor(meta_val_s, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy().squeeze()

ms_ridge = msape(y_val, ridge_val)
ms_gb = msape(y_val, gb_val)
ms_mlp = msape(y_val, mlp_val)

print("\n" + "="*60)
print("Holdout validation MSAPE:")
print(f"  Ridge      : {ms_ridge:.3f}")
print(f"  GradBoosting: {ms_gb:.3f}")
print(f"  Meta-MLP   : {ms_mlp:.3f}")
print("="*60)

# Weighted ensemble
weights = np.array([
    1.0 / (ms_ridge + 1e-6),
    1.0 / (ms_gb + 1e-6),
    1.0 / (ms_mlp + 1e-6)
])
weights /= weights.sum()
print(f"\nWeighted ensemble: Ridge({weights[0]:.2f}) + GB({weights[1]:.2f}) + MLP({weights[2]:.2f})")

ensemble_val = weights[0] * ridge_val + weights[1] * gb_val + weights[2] * mlp_val
ms_ensemble = msape(y_val, ensemble_val)
print(f"Weighted Ensemble MSAPE: {ms_ensemble:.3f}")

best_name = "weighted_ensemble"
best_model_score = ms_ensemble

print(f"\n✅ Selected: {best_name.upper()} (MSAPE: {best_model_score:.3f})")

# ============= SAVE ARTIFACTS =============

print(f"\nSaving artifacts...")
torch.save(img_full.state_dict(), f"{ARTIFACT_DIR}/image_only_model.pt")
torch.save(txt_full.state_dict(), f"{ARTIFACT_DIR}/text_only_model.pt")
dump(meta_scaler, f"{ARTIFACT_DIR}/meta_scaler.joblib")
dump(ridge_cv, f"{ARTIFACT_DIR}/ridge_model.joblib")
dump(gb, f"{ARTIFACT_DIR}/gb_model.joblib")
torch.save(meta_mlp.state_dict(), f"{ARTIFACT_DIR}/meta_mlp.pt")

save_config = {
    "best_ensemble": best_name,
    "weights": weights.tolist(),
    "validation_scores": {
        "ridge": float(ms_ridge),
        "gb": float(ms_gb),
        "mlp": float(ms_mlp),
        "weighted": float(ms_ensemble)
    },
    "target_is_log": bool(target_is_log),
    "img_dim": int(Xi_s.shape[1]),
    "txt_dim": int(Xt_s.shape[1]),
    "paths": {
        "img_model": f"{ARTIFACT_DIR}/image_only_model.pt",
        "txt_model": f"{ARTIFACT_DIR}/text_only_model.pt",
        "meta_scaler": f"{ARTIFACT_DIR}/meta_scaler.joblib",
        "ridge": f"{ARTIFACT_DIR}/ridge_model.joblib",
        "gb": f"{ARTIFACT_DIR}/gb_model.joblib",
        "meta_mlp": f"{ARTIFACT_DIR}/meta_mlp.pt",
        "pretrained_pt": PRETRAINED_PT
    }
}

with open(f"{ARTIFACT_DIR}/ensemble_config.json", "w") as f:
    json.dump(save_config, f, indent=2)

print("✅ Artifacts saved!")

# ============================================================
# INFERENCE ON TEST SET
# ============================================================
print("\n" + "="*60)
print("Starting inference on test set...")

test_df = pd.read_csv(TEST_CSV)
test_sample_ids = test_df['sample_id'].values if 'sample_id' in test_df.columns else np.arange(len(test_df))

# Parse encodings
print("Parsing test encodings...")
test_df['image_encoding'] = test_df['image_encoding'].apply(safe_parse)
test_df['text_encoding'] = test_df['text_encoding'].apply(safe_parse)

missing_img = test_df['image_encoding'].isna().sum()
missing_txt = test_df['text_encoding'].isna().sum()
print(f"Missing image encodings: {missing_img}")
print(f"Missing text encodings: {missing_txt}")

# Get dimensions
first_valid_img = test_df['image_encoding'].dropna().iloc[0]
first_valid_txt = test_df['text_encoding'].dropna().iloc[0]
img_dim = len(first_valid_img)
txt_dim = len(first_valid_txt)

# Fill missing
def fill_missing(x, dim):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.zeros(dim)
    return x

test_df['image_encoding'] = test_df['image_encoding'].apply(lambda x: fill_missing(x, img_dim))
test_df['text_encoding'] = test_df['text_encoding'].apply(lambda x: fill_missing(x, txt_dim))

Xi_te = np.asarray(test_df['image_encoding'].tolist(), dtype=np.float32)
Xt_te = np.asarray(test_df['text_encoding'].tolist(), dtype=np.float32)

print(f"Test image shape: {Xi_te.shape}")
print(f"Test text shape: {Xt_te.shape}")

# Scale
Xi_te_s = scaler_img.transform(Xi_te)
Xt_te_s = scaler_txt.transform(Xt_te)

# Generate predictions
print("Generating test predictions...")
img_pred_log = predict_uni_log(img_full, Xi_te_s)
txt_pred_log = predict_uni_log(txt_full, Xt_te_s)
mul_pred_log = predict_multi_log(multi, Xi_te_s, Xt_te_s)

img_p = np.expm1(img_pred_log)
txt_p = np.expm1(txt_pred_log)
mul_p = np.expm1(mul_pred_log) if target_is_log else mul_pred_log

# Create meta features
meta_test = create_advanced_meta_features(img_p, txt_p, mul_p)
meta_test_s = meta_scaler.transform(meta_test)

# Generate predictions from all models
ridge_p = ridge_cv.predict(meta_test_s)
gb_p = gb.predict(meta_test_s)
mlp_p = meta_mlp(torch.tensor(meta_test_s, dtype=torch.float32, device=DEVICE)).detach().cpu().numpy().squeeze()

# Weighted ensemble
final_preds = weights[0] * ridge_p + weights[1] * gb_p + weights[2] * mlp_p

# Ensure non-negative
final_preds = np.maximum(final_preds, 0.0)

# Save predictions
out_df = pd.DataFrame({
    "sample_id": test_sample_ids,
    "price": final_preds
})

OUT_CSV = f"{BASE_DIR}/ensemble_final_predictions5.csv"
out_df.to_csv(OUT_CSV, index=False)

print(f"\n✅ Predictions saved to: {OUT_CSV}")
print(f"Total test samples: {len(out_df)}")
print(f"Price range: ${final_preds.min():.2f} - ${final_preds.max():.2f}")
print(f"Mean price: ${final_preds.mean():.2f}")
print(f"Median price: ${np.median(final_preds):.2f}")
print(f"Std price: ${final_preds.std():.2f}")
print("="*60)
print("\nFirst 10 predictions:")
print(out_df.head(10))
print("\nLast 10 predictions:")
print(out_df.tail(10))

Loading training data...
Loading pretrained model...
Pretrained model loaded!

Generating OOF predictions (5-fold)...
  Fold 1/5...
  Fold 2/5...
  Fold 3/5...
  Fold 4/5...
  Fold 5/5...

Creating advanced meta features...
Meta features shape: (63749, 16)
Training Ridge ensemble...
OOF Ridge MSAPE: 3.193
Training GradientBoosting ensemble...
OOF GradientBoosting MSAPE: 3.000
Training Enhanced Meta-MLP...


Train Meta-MLP: 100%|██████████| 600/600 [00:09<00:00, 65.30it/s]


OOF Meta-MLP MSAPE: 86.639

Evaluating on validation split...

Holdout validation MSAPE:
  Ridge      : 36.944
  GradBoosting: 36.954
  Meta-MLP   : 90.287

Weighted ensemble: Ridge(0.42) + GB(0.42) + MLP(0.17)
Weighted Ensemble MSAPE: 38.779

✅ Selected: WEIGHTED_ENSEMBLE (MSAPE: 38.779)

Saving artifacts...
✅ Artifacts saved!

Starting inference on test set...
Parsing test encodings...
Missing image encodings: 1
Missing text encodings: 0
Test image shape: (75000, 512)
Test text shape: (75000, 512)
Generating test predictions...

✅ Predictions saved to: /content/drive/MyDrive//ensemble_final_predictions5.csv
Total test samples: 75000
Price range: $0.00 - $627.38
Mean price: $17.55
Median price: $12.36
Std price: $18.40

First 10 predictions:
   sample_id      price
0     100179  10.553210
1     245611  10.630486
2     146263  13.405149
3      95658   4.497013
4      36806  18.822765
5     148239   3.719091
6      92659  13.047042
7       3780   7.411662
8     196940  16.190791
9      